module load anaconda3/3.11
source activate /home/sofias/software/envs/tools
alias claude-personal='CLAUDE_CONFIG_DIR=$HOME/.claude-personal claude'
source ~/.bashrc


In [ ]:
import kagglehub

In [ ]:
path = kagglehub.competition_download('umud-challenge-muscle-architecture-in-ultrasound-data')

print("Path to competition files:", path)

# Exploring the UMUD challenge data

**What is this dataset?** The *UMUD challenge — Muscle Architecture in Ultrasound Data* contains B-mode ultrasound
images of skeletal muscle, scanned lengthwise along the muscle. In these images you can see:

* **Aponeuroses**: the bright, thick bands of connective tissue at the top (superficial) and bottom (deep) of the muscle.
* **Fascicles**: the thin, slanted bright lines between the two aponeuroses. They are bundles of muscle fibres.

The competition asks for three **muscle architecture** measurements per test image (see `sample_submission.csv`):

| Column | Meaning | Unit |
|---|---|---|
| `pa_deg` | **Pennation angle**: angle between the fascicles and the deep aponeurosis | degrees |
| `fl_mm` | **Fascicle length**: length of a fascicle from one aponeurosis to the other | mm |
| `mt_mm` | **Muscle thickness**: distance between the superficial and deep aponeurosis | mm |

The training data does **not** include these numbers. Instead it provides segmentation masks, so the usual approach is:
1. train one model to segment the aponeuroses and another to segment the fascicles,
2. then calculate thickness, angle and length geometrically from the predicted masks.

| Folder | Contents |
|---|---|
| `apo_imgs_v1` / `apo_masks_v1` | ultrasound images + binary masks of the **aponeuroses** |
| `fasc_imgs_v1` / `fasc_masks_v1` | ultrasound images + binary masks of the **fascicles** |
| `test_images_v2` | unlabelled test images to predict on |

In [ ]:
import os, glob, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

# `path` comes from the kagglehub download cell above; fall back to the project's `data` symlink
DATA = path if "path" in globals() else "data"

DIRS = {
    "apo_imgs":  os.path.join(DATA, "apo_imgs_v1/apo_images_new_model_v1"),
    "apo_masks": os.path.join(DATA, "apo_masks_v1/apo_masks_new_model_v1"),
    "fasc_imgs": os.path.join(DATA, "fasc_imgs_v1/fasc_images_new_model_v1"),
    "fasc_masks": os.path.join(DATA, "fasc_masks_v1/fasc_masks_new_model_v1"),
    "test_imgs": os.path.join(DATA, "test_images_v2/test_set_v2"),
}
IMG_EXT = (".tif", ".tiff", ".png", ".jpg")

def list_images(d):
    # Thumbs.db files are Windows thumbnail caches, not data, so they are skipped
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))

files = {k: list_images(d) for k, d in DIRS.items()}

def load_img(p):
    # Pillow is used because some TIFFs are LZW-compressed
    return np.array(Image.open(p).convert("RGB"))

def load_mask(p):
    # some masks are palette ("P") images, so convert to grayscale before thresholding
    return np.array(Image.open(p).convert("L")) > 127

## 1. File inventory
How many files are in each folder, which formats they use, and whether every image has a mask with the same filename.

In [ ]:
inventory = pd.DataFrame([
    {"folder": k, "n_files": len(v),
     "extensions": dict(Counter(os.path.splitext(f)[1].lower() for f in v))}
    for k, v in files.items()
])
display(inventory)

for kind in ["apo", "fasc"]:
    imgs, masks = set(files[f"{kind}_imgs"]), set(files[f"{kind}_masks"])
    print(f"{kind}: {len(imgs & masks)} matched pairs, "
          f"{len(imgs - masks)} images without mask, {len(masks - imgs)} masks without image")

## 2. Image sizes and formats
This reads only the file headers, so it is fast even for ~7,900 files.

In [ ]:
def header_info(d, names):
    rows = []
    for n in names:
        with Image.open(os.path.join(d, n)) as im:
            rows.append({"file": n, "width": im.width, "height": im.height, "mode": im.mode})
    return pd.DataFrame(rows)

meta = {k: header_info(DIRS[k], v) for k, v in files.items()}

summary = []
for k, df in meta.items():
    summary.append({
        "folder": k, "n": len(df),
        "width min/median/max": f"{df.width.min()} / {int(df.width.median())} / {df.width.max()}",
        "height min/median/max": f"{df.height.min()} / {int(df.height.median())} / {df.height.max()}",
        "n distinct sizes": df.groupby(["width", "height"]).ngroups,
        "modes": dict(df["mode"].value_counts()),
    })
display(pd.DataFrame(summary))

print("Most common image sizes (width x height):")
for k in ["apo_imgs", "fasc_imgs", "test_imgs"]:
    top = meta[k].groupby(["width", "height"]).size().sort_values(ascending=False).head(5)
    print(f"  {k}: " + ", ".join(f"{w}x{h} ({n})" for (w, h), n in top.items()))

### Do images and masks have the same size?
For segmentation, every mask should cover the same pixels as its image. The check below compares their sizes.

In [ ]:
for kind in ["apo", "fasc"]:
    pair = meta[f"{kind}_imgs"].merge(meta[f"{kind}_masks"], on="file", suffixes=("_img", "_mask"))
    same = (pair.width_img == pair.width_mask) & (pair.height_img == pair.height_mask)
    print(f"{kind}: {same.sum()} / {len(pair)} pairs have identical size")
    print("   image sizes of the mismatched pairs:",
          dict(pair.loc[~same].groupby(["width_img", "height_img"]).size()))

**Note:** for every 1200×800 image, the mask has a different size *and aspect ratio* (e.g. 1152×864 or 660×556).
The images seem to have been exported onto a fixed 1200×800 canvas, while the masks keep the original scan size.
In the overlays below, **stretching the mask to the image size makes it line up with the anatomy**, so these pairs can be
used. Just remember to **resize each mask to its image (nearest-neighbour)** when you build the training set.

## 3. Example images
Each row shows: the ultrasound image, its binary mask, and the mask drawn over the image in red.

In [ ]:
random.seed(0)

def show_pairs(kind, n=4):
    names = random.sample(sorted(set(files[f"{kind}_imgs"]) & set(files[f"{kind}_masks"])), n)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
    for row, name in zip(axes, names):
        img = load_img(os.path.join(DIRS[f"{kind}_imgs"], name))
        mask = load_mask(os.path.join(DIRS[f"{kind}_masks"], name))
        mask_rs = Image.fromarray(mask).resize((img.shape[1], img.shape[0]), Image.NEAREST)
        if kind == "fasc":  # fascicle lines are 1-2 px wide: thicken them for display only
            mask_rs = mask_rs.convert("L").filter(ImageFilter.MaxFilter(5))
        mask_rs = np.array(mask_rs) > 0
        row[0].imshow(img); row[0].set_title(f"{name}  image {img.shape[1]}x{img.shape[0]}")
        row[1].imshow(mask, cmap="gray"); row[1].set_title(f"mask {mask.shape[1]}x{mask.shape[0]}")
        row[2].imshow(img); row[2].imshow(np.ma.masked_where(~mask_rs, mask_rs), cmap="autumn", alpha=0.6)
        row[2].set_title("overlay (mask resized to image)" + (", lines thickened" if kind == "fasc" else ""))
        for ax in row: ax.axis("off")
    fig.suptitle(f"{kind.upper()} training pairs", fontsize=16)
    plt.tight_layout(); plt.show()

show_pairs("apo")

**Aponeurosis masks** mark the two bright bands that enclose the muscle. The vertical gap between them is the
**muscle thickness**.

In [ ]:
show_pairs("fasc")

**Fascicle masks** mark only a few clearly visible fascicle segments, not every fibre. A model trained on them learns
to find fascicle *orientation*. The fascicle length is then usually extrapolated between the two aponeuroses, because
fascicles often extend beyond the image.

In [ ]:
names = random.sample(files["test_imgs"], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, n in zip(axes.flat, names):
    img = load_img(os.path.join(DIRS["test_imgs"], n))
    ax.imshow(img); ax.set_title(f"{n}  {img.shape[1]}x{img.shape[0]}"); ax.axis("off")
fig.suptitle("Test images (unlabelled)", fontsize=16)
plt.tight_layout(); plt.show()

## 4. Pixel statistics
Computed on a random sample of 200 images per folder, to keep it quick:
* **brightness**: mean grey value (0–255) and contrast (standard deviation) of each image
* **mask coverage**: share of mask pixels that are foreground. This shows how *imbalanced* the segmentation task is.

In [ ]:
N = 200
rows = []
for k in ["apo_imgs", "fasc_imgs", "test_imgs"]:
    for n in random.sample(files[k], min(N, len(files[k]))):
        g = np.array(Image.open(os.path.join(DIRS[k], n)).convert("L"), dtype=np.float32)
        rows.append({"set": k, "mean": g.mean(), "std": g.std()})
pix = pd.DataFrame(rows)
display(pix.groupby("set")[["mean", "std"]].describe().round(1))

cov = []
for k in ["apo_masks", "fasc_masks"]:
    for n in random.sample(files[k], min(N, len(files[k]))):
        cov.append({"set": k, "foreground_%": 100 * load_mask(os.path.join(DIRS[k], n)).mean()})
cov = pd.DataFrame(cov)
display(cov.groupby("set")["foreground_%"].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for k, g in pix.groupby("set"):
    axes[0].hist(g["mean"], bins=30, alpha=0.5, label=k)
    axes[1].hist(g["std"], bins=30, alpha=0.5, label=k)
axes[0].set_title("Mean brightness per image"); axes[0].legend()
axes[1].set_title("Contrast (std) per image"); axes[1].legend()
for k, g in cov.groupby("set"):
    axes[2].hist(g["foreground_%"], bins=30, alpha=0.6, label=k)
axes[2].set_title("Mask foreground (% of pixels)"); axes[2].legend()
plt.tight_layout(); plt.show()

## 5. Submission format

In [ ]:
sub = pd.read_csv(os.path.join(DATA, "sample_submission.csv"), sep=";", encoding="utf-8-sig")
display(sub)
print("Test images to predict:", len(files["test_imgs"]))

## Key takeaways
* **Two segmentation tasks**: aponeuroses (~1,050 pairs) and fascicles (~2,760 pairs). The final answer (angle,
  length, thickness) is calculated from the segmentations, not predicted directly.
* **Mixed sizes and scanners**: images come in many resolutions and screen layouts (text, colour bars, borders),
  so they need resizing to a fixed input size, and probably cropping to the actual scan area.
* **Image/mask size mismatch**: every 1200×800 image (732 apo, 1,714 fasc) has a mask stored at another resolution.
  Stretching the mask to the image aligns it, so resize masks to their images when loading.
* **Strong class imbalance**, especially for fascicles (foreground is often well below 1% of pixels). Use
  Dice/IoU-style losses instead of plain pixel accuracy.
* **Units**: `fl_mm` and `mt_mm` are in millimetres, so pixel distances must be converted with the image's pixel
  spacing (depth scale). This is not given directly in the data and must be estimated or read from the image.
* The sample submission uses **`;`** as separator, and the test set mixes `.tif` and `.png` files.